# FAISS indexing and retrieval inspection

Goal:
- load saved chunk embeddings and chunk metadata
- verify that embeddings and metadata are aligned
- confirm whether embeddings are normalized
- build one FAISS index per chunk size
- retrieve top-k chunks for a question
- inspect retrieval behavior qualitatively (To check if any previous step is broken) 

Important:
- this notebook implements the retrieval stage only


In [1]:
import json
import faiss
import numpy as np
from pathlib import Path

data_dir = Path("/home/a/arfaoui/rag_project/data")
chunk_sizes = [32, 128, 256]

In [2]:
# Load saved embeddings and metadata for each chunk size
# We keep both because FAISS needs vectors, and retrieval later needs metadata mapping.

all_embeddings = {}
all_metadata = {}

for size in chunk_sizes:
    # Load embedding matrix
    emb_path = data_dir / f"embeddings_{size}.npy"
    embeddings = np.load(emb_path)
    all_embeddings[size] = embeddings

    # Load chunk metadata from JSON file
    meta_path = data_dir / f"chunks_metadata_{size}.json"
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    all_metadata[size] = metadata

    # Print quick sanity checks
    print(f"\nchunk_size={size}")
    print(f"Embeddings shape: {embeddings.shape}")
    print(f"Metadata entries: {len(metadata)}")


chunk_size=32
Embeddings shape: (292199, 384)
Metadata entries: 292199

chunk_size=128
Embeddings shape: (96686, 384)
Metadata entries: 96686

chunk_size=256
Embeddings shape: (69845, 384)
Metadata entries: 69845


In [3]:
# Check if embeddings are normalized or not 
# This matters because FAISS IndexFlatIP corresponds to cosine similarity only when vectors are normalized.

sample = all_embeddings[128][:10]
all = all_embeddings[128]
sample_check = np.linalg.norm(sample, axis=1)
norms = np.linalg.norm(all, axis=1)

print(sample_check)
print(f"Mean norm all embeddings: {np.mean(norms):.4f}")
print(f"Min norm all embeddings: {np.min(norms):.4f}")
print(f"Max norm all embeddings: {np.max(norms):.4f}")

[0.99999994 0.99999994 0.99999994 1.         1.0000001  0.99999994
 1.         1.         1.         0.99999994]
Mean norm all embeddings: 1.0000
Min norm all embeddings: 1.0000
Max norm all embeddings: 1.0000


**Observations**
- From the inspection, the embeddings appear to be normalized as the overall min, max, and mean values across all embedding vectors are approximately 1.0.
- Normalization is a crucial step. Skipping it introduces hidden bias if embeddings were not normalized:  
  • Vectors with larger magnitudes will dominate the similarity scores  
  • Retrieval quality degrades because comparisons become scale‑dependent

Note:
 - all embedings (32 and 256) are checked 


In [4]:
# Build FAISS indexes for each chunk size
# We use IndexFlatIP because embeddings are normalized → cosine similarity

faiss_indexes = {}

for size, embeddings in all_embeddings.items():
    
    print(f"\nBuilding FAISS index for chunk_size={size} ...")
    
    dim = embeddings.shape[1]  # embedding dimension (384)
    
    # Create FAISS index 
    # Flat inner-product index
    # Works as cosine similarity because embeddings are L2-normalized
    index = faiss.IndexFlatIP(dim)  
    
    # Add embeddings (Vectors) to index
    # FAISS expects float32 (From GitHub documentation)
    index.add(embeddings.astype("float32"))
    
    faiss_indexes[size] = index
    
    # Print index info
    print(f"Index size: {index.ntotal}")
    # Now vector_id → chunk


Building FAISS index for chunk_size=32 ...
Index size: 292199

Building FAISS index for chunk_size=128 ...
Index size: 96686

Building FAISS index for chunk_size=256 ...
Index size: 69845


In [5]:
# Optional
# Save FAISS indexes to disk so they can be reused without rebuilding later if needed

for size, index in faiss_indexes.items():
    index_path = data_dir / f"faiss_index_{size}.index"
    faiss.write_index(index, str(index_path))
    print(f"Saved FAISS index for chunk_size={size}: {index_path}")

Saved FAISS index for chunk_size=32: /home/a/arfaoui/rag_project/data/faiss_index_32.index
Saved FAISS index for chunk_size=128: /home/a/arfaoui/rag_project/data/faiss_index_128.index
Saved FAISS index for chunk_size=256: /home/a/arfaoui/rag_project/data/faiss_index_256.index


In [7]:
# Reload the saved sample correctly from JSON Lines format
# The saved sample file is in JSON Lines format, so we load it with the datasets library.

from datasets import load_dataset

sample_path = data_dir / "hotpotqa_full_sample.json"

hotpot_sample = load_dataset("json", data_files=str(sample_path))["train"]

print("Loaded questions:", len(hotpot_sample))
print("Example question:", hotpot_sample[0]["question"])
print("Example answer:", hotpot_sample[0]["answer"])

Loaded questions: 7405
Example question: Were Scott Derrickson and Ed Wood of the same nationality?
Example answer: yes


In [8]:
# Load the embedding model used for both chunk and query embeddings
# This ensures that question and chunk vectors are in the same vector space.

from sentence_transformers import SentenceTransformer

modelName = "BAAI/bge-small-en-v1.5"
embed_model = SentenceTransformer(modelName)

print("Embedding model loaded:", modelName)

Embedding model loaded: BAAI/bge-small-en-v1.5


In [9]:
# Function to embed a query and retrieve top-k chunks from FAISS

def retrieve_top_k(question, index, metadata, k=5):
    """
    Given a question, retrieve top-k most similar chunks.
    
    Returns:
    - retrieved_chunks: list of metadata entries
    - scores: similarity scores
    """
    
    # BGE models expect "query: " prefix for queries
    query_text = "query: " + question
    
    # Encode query
    query_embedding = embed_model.encode([query_text])
    
    # Convert to float32 (FAISS requirement)
    query_embedding = np.array(query_embedding).astype("float32")
    
    # Search in FAISS index
    D, I = index.search(query_embedding, k)
    
    # Retrieve corresponding chunks
    retrieved_chunks = [metadata[idx] for idx in I[0]]
    scores = D[0]
    
    return retrieved_chunks, scores

In [10]:
# Test retrieval on one example question

example = hotpot_sample[0]

question = example["question"]
correct_answer = example["answer"]

print("Question:", question)
print("Correct_answer:", correct_answer)

# Choose one chunk size 
size = 128

index = faiss_indexes[size]
metadata = all_metadata[size]


chunks, scores = retrieve_top_k(question, index, metadata, k=5)

print("\n--- Retrieved Chunks ---\n")

# Print retrieved chunks more clearly for manual inspection

for i, (chunk, score) in enumerate(zip(chunks, scores)):
    print(f"[{i+1}] Score: {score:.4f}")
    print("Title:", chunk["title"])
    print("Text preview:", chunk["text"][:300])
    print("-" * 60)

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Correct_answer: yes

--- Retrieved Chunks ---

[1] Score: 0.6869
Title: Scott Derrickson
Text preview: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe ins
------------------------------------------------------------
[2] Score: 0.6579
Title: Ed Wood
Text preview: Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.
------------------------------------------------------------
[3] Score: 0.6361
Title: Ed Wood (film)
Text preview: Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the 

In [11]:
# Compare retrieval across chunk sizes for the same question
# This helps us qualitatively inspect how chunk size changes the retrieved results.
# Part of the learning process and experience process 
# Try to inspect the differences based on the chunk size 

example = hotpot_sample[0]
question = example["question"]
correct_answer = example["answer"]

print("Question:", question)
print("Correct answer:", correct_answer)



for size in [32, 128, 256]:
    print(f"\n===== Retrieval with chunk_size={size} =====\n")
    
    index = faiss_indexes[size]
    metadata = all_metadata[size]
    
    chunks, scores = retrieve_top_k(question, index, metadata, k=5)
    
    for i, (chunk, score) in enumerate(zip(chunks, scores)):
        print(f"[{i+1}] Score: {score:.4f}")
        print("Title:", chunk["title"])
        print("Text preview:", chunk["text"][:250])
        print("-" * 60)

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Correct answer: yes

===== Retrieval with chunk_size=32 =====

[1] Score: 0.7431
Title: Scott Derrickson
Text preview: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.
------------------------------------------------------------
[2] Score: 0.6585
Title: Adam Collis
Text preview: first work was the assistant director for the Scott Derrickson's short "Love in the Ruins" (1995).  In 1998, he played
------------------------------------------------------------
[3] Score: 0.6405
Title: Elijah Wood
Text preview: Elijah Jordan Wood (born January 28, 1981) is an American actor, voice actor, DJ, and producer.  He is best known
------------------------------------------------------------
[4] Score: 0.6289
Title: Ed Wood
Text preview: Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer
-------

# Observation:
For the Kim Hyang-gi question:  
- the retriever consistently surfaces chunks about Kim Hyang-gi it also retrieves the Wedding Dress chunk  
- We can already see the chunk size effect visually: chunk_size=32 returns narrow snippets, chunk_size=128 and 256 return progressively richer context
- chunk_size=32: Returns fragmented, narrow snippets. The relevant date chunk (Wedding Dress) appears but ranked 4th (score 0.6874). Lots of noise: unrelated Kim films dominate the top spots.
- chunk_size=128: Much better. The biography chunk with full career context comes 1st (score 0.7350), and the Wedding Dress chunk is 2nd (0.6874). Richer context, better ranking.
- chunk_size=256: Nearly identical to 128 in ranking and scores. Top chunks are the same, just slightly longer passages.


# What the printed score means:
- the score is a retrieval similarity score returned by FAISS
- it reflects how similar a retrieved chunk is to the question in embedding space
- it is not a QA metric and should not be confused with F1 or exact match

Why this notebook matters:
- it validates the retrieval stage before involving the LLM
- it helps inspect how chunk size affects retrieved context
- it prepares the pipeline for the next notebook, where retrieved chunks will be passed to the LLM for answer generation and evaluation (Very important Step)

In [12]:
# Build a single text context string from retrieved chunks
# This will later be passed into the LLM prompt, without it the retrieved chunks are useful information but can't give it to the model
# This makes it:
#  -human-readable
#  -structured for the LLM
#  -easy to debug


def build_retrieved_context(retrieved_chunks):
    """
    Concatenate retrieved chunks into one context string.
    """
    parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        parts.append(f"[Chunk {i}] Title: {chunk['title']}\n{chunk['text']}")

    return "\n\n".join(parts)

In [13]:
context_text = build_retrieved_context(chunks)
print(context_text[:1000])

[Chunk 1] Title: Scott Derrickson
Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."

[Chunk 2] Title: Ed Wood
Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.

[Chunk 3] Title: Ed Wood (film)
Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.

[Chunk 4] Title: Ade Edmo